# Tarea 2: Reducción de Dimensionalidad con PCA
## Máster en IA Aplicada a la Ciberseguridad
### Módulo 2: Fundamentos de Machine Learning

---

## Objetivos
1. **Comprender PCA**: Reducir dimensionalidad manteniendo máxima varianza
2. **Implementar desde cero**: Calcular autovalores y autovectores de matriz de covarianza
3. **Visualizar y analizar**: Graficar datos reducidos y analizar varianza explicada

## Dataset
- **Iris Dataset**: 150 muestras, 4 features, 3 clases
- **Objetivo**: Reducir de 4D a 2D

## 1. Importar librerías y configuración

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Configuración de visualización
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Añadir src al path
sys.path.insert(0, os.path.abspath('..'))

from src.pca_manual import PCAManual
from src.data_loader import *
from src.visualization import *

print("✓ Librerías importadas correctamente")

## 2. Cargar y explorar datos

In [ ]:
# Cargar dataset Iris
X, y, feature_names, target_names = load_iris_data()

print(f"Shape de datos: {X.shape}")
print(f"Features: {feature_names}")
print(f"Clases: {target_names}")

# Crear DataFrame para exploración
df = create_dataframe(X, y, feature_names, target_names)
df.head(10)

In [ ]:
# Estadísticas descriptivas
stats = get_data_statistics(X, feature_names)
print("📊 Estadísticas del dataset:")
stats

In [ ]:
# Distribución de clases
print("Distribución de clases:")
print(df['species'].value_counts())

## 3. Preprocesamiento: Estandarización

⚠️ **IMPORTANTE**: La estandarización es fundamental antes de aplicar PCA porque:
- PCA es sensible a la escala de las variables
- Features con mayor varianza dominarían los componentes principales
- Queremos que todas las features contribuyan equitativamente

In [ ]:
# Estandarizar datos: media=0, std=1
X_scaled, scaler = standardize_data(X)

print("Antes de estandarización:")
print(f"  Media: {np.mean(X, axis=0)}")
print(f"  Std: {np.std(X, axis=0)}")

print("\nDespués de estandarización:")
print(f"  Media: {np.mean(X_scaled, axis=0)}")
print(f"  Std: {np.std(X_scaled, axis=0)}")

## 4. Análisis de Correlación

In [ ]:
# Matriz de correlación
corr_matrix = get_correlation_matrix(X, feature_names)
plot_correlation_heatmap(corr_matrix)

## 5. PCA Manual (Implementación desde cero)

### Algoritmo:
1. **Centrar los datos**: Restar la media
2. **Calcular matriz de covarianza**: $\text{Cov}(X) = \frac{1}{n-1} X^T X$
3. **Calcular autovalores y autovectores** de la matriz de covarianza
4. **Ordenar** por autovalores descendentes
5. **Seleccionar** los primeros k componentes
6. **Proyectar** los datos: $X_{\text{pca}} = X \cdot V_k$

In [ ]:
# Aplicar PCA manual
pca_manual = PCAManual(n_components=2)
X_pca_manual = pca_manual.fit_transform(X_scaled)

print("✓ PCA manual completado")
print(f"\nShape de datos transformados: {X_pca_manual.shape}")

In [ ]:
# Autovalores y Autovectores
print("📈 AUTOVALORES (Varianzas):")
print(pca_manual.eigenvalues_)

print("\n📊 AUTOVECTORES (Direcciones principales):")
print(pca_manual.eigenvectors_)

In [ ]:
# Varianza explicada por cada componente
print("📊 Varianza explicada por componente:")
for i, var_ratio in enumerate(pca_manual.explained_variance_ratio_):
    print(f"  PC{i+1}: {var_ratio*100:.2f}%")

print(f"\nVarianza total capturada: {np.sum(pca_manual.explained_variance_ratio_)*100:.2f}%")

In [ ]:
# Matriz de covarianza
cov_matrix = pca_manual.get_covariance_matrix(X_scaled)
print("Matriz de Covarianza:")
plot_covariance_matrix(cov_matrix, feature_names)

## 6. Visualización de resultados

In [ ]:
# Gráfico de dispersión 2D
plot_pca_scatter(
    X_pca_manual, y, target_names,
    title="PCA Manual - Dataset Iris en 2D"
)

In [ ]:
# Varianza explicada (todos los componentes)
pca_all = PCAManual(n_components=4)
pca_all.fit(X_scaled)

plot_variance_explained(
    pca_all.explained_variance_ratio_,
    cumulative=True
)

In [ ]:
# Biplot: Datos + Vectores de features
plot_biplot(
    X_pca_manual, y,
    pca_manual.components_,
    feature_names,
    target_names
)

## 7. Comparación con sklearn

In [ ]:
# PCA con sklearn
pca_sklearn = PCA(n_components=2)
X_pca_sklearn = pca_sklearn.fit_transform(X_scaled)

print("📊 Varianza explicada (sklearn):")
for i, var_ratio in enumerate(pca_sklearn.explained_variance_ratio_):
    print(f"  PC{i+1}: {var_ratio*100:.2f}%")

In [ ]:
# Comparar resultados
print("🔍 Comparación Manual vs Sklearn:")
print("\nDiferencia en varianza explicada:")
diff = np.abs(pca_manual.explained_variance_ratio_ - pca_sklearn.explained_variance_ratio_)
for i, d in enumerate(diff):
    print(f"  PC{i+1}: {d:.8f}")

print("\n✓ Las implementaciones son equivalentes (diferencias numéricas despreciables)")

In [ ]:
# Visualizar comparación
plot_comparison_manual_vs_sklearn(
    X_pca_manual, X_pca_sklearn,
    y, target_names
)

## 8. Análisis e Interpretación

### 8.1 Distribución de clases en espacio reducido

In [ ]:
# Análisis por clase
df_pca = pd.DataFrame(X_pca_manual, columns=['PC1', 'PC2'])
df_pca['species'] = [target_names[i] for i in y]

print("Estadísticas por clase en espacio PCA:")
print(df_pca.groupby('species').describe())

### 8.2 Componentes principales y loadings

In [ ]:
# Componentes (combinaciones lineales de features originales)
components_df = pd.DataFrame(
    pca_manual.components_.T,
    columns=['PC1', 'PC2'],
    index=feature_names
)

print("Componentes Principales:")
print(components_df)

# Visualizar
components_df.plot(kind='bar', figsize=(10, 6))
plt.title('Loadings de Features en Componentes Principales')
plt.xlabel('Features')
plt.ylabel('Loading')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Componente')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Reflexión y Conclusiones

### ¿Cómo se distribuyen las clases?
- **Setosa**: Claramente separada de las otras dos especies
- **Versicolor y Virginica**: Tienen cierto solapamiento pero son distinguibles
- Los 2 componentes capturan >95% de la varianza, por lo que la representación 2D es muy fiel a los datos originales

### Proporción de varianza capturada
- **PC1**: Captura ~72-73% de la varianza total
- **PC2**: Captura ~22-23% de la varianza total
- **Total**: ~95-96% de la varianza con solo 2 componentes
- Se pierde solo ~4-5% de información al reducir de 4D a 2D

### Interpretación de componentes
- PC1 está fuertemente correlacionado con petal length y petal width
- PC2 tiene mayor contribución de sepal length y sepal width
- Los componentes capturan las direcciones de máxima variabilidad en los datos

## 10. Reconstrucción de datos

In [ ]:
# Reconstruir datos originales desde espacio PCA
X_reconstructed = pca_manual.inverse_transform(X_pca_manual)

# Calcular error de reconstrucción
reconstruction_error = np.mean(np.square(X_scaled - X_reconstructed))
print(f"Error de reconstrucción (MSE): {reconstruction_error:.6f}")

# Comparar muestra original vs reconstruida
sample_idx = 0
print(f"\nMuestra {sample_idx}:")
print(f"  Original:      {X_scaled[sample_idx]}")
print(f"  Reconstruido:  {X_reconstructed[sample_idx]}")
print(f"  Diferencia:    {X_scaled[sample_idx] - X_reconstructed[sample_idx]}")